# 02 Feature Engineering

This notebook creates the first set of pre-game features for the NFL forecasting model.

The main goal is to calculate team statistics that would have been available before each game was played. This helps avoid data leakage and makes the dataset realistic for prediction.

In [24]:
# Import packages
import pandas as pd

In [25]:
# Load the processed game results
games = pd.read_csv("../data/processed/game_results_2023_2025.csv")

games.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0


In [26]:
# Check the data
games.shape
games.columns.tolist()
games.head(10)

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0
5,2023,1,2023_01_TB_MIN,2023-09-10,MIN,TB,17.0,20.0,0,-3.0
6,2023,1,2023_01_TEN_NO,2023-09-10,NO,TEN,16.0,15.0,1,1.0
7,2023,1,2023_01_SF_PIT,2023-09-10,PIT,SF,7.0,30.0,0,-23.0
8,2023,1,2023_01_ARI_WAS,2023-09-10,WAS,ARI,20.0,16.0,1,4.0
9,2023,1,2023_01_GB_CHI,2023-09-10,CHI,GB,20.0,38.0,0,-18.0


In [27]:
# Sort games by season, week, and date
games["gameday"] = pd.to_datetime(games["gameday"])

games = games.sort_values(["season", "week", "gameday"]).reset_index(drop=True)

games.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0


## Create Team-Game Rows

To calculate team statistics, each game will be converted into two rows: one for the home team and one for the away team.

In [28]:
home_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "home_team",
        "away_team",
        "home_score",
        "away_score"
    ]
].copy()

home_rows = home_rows.rename(
    columns={
        "home_team": "team",
        "away_team": "opponent",
        "home_score": "points_scored",
        "away_score": "points_allowed"
    }
)

home_rows["is_home"] = 1

In [29]:
# Create away team rows
away_rows = games[
    [
        "season",
        "week",
        "game_id",
        "gameday",
        "away_team",
        "home_team",
        "away_score",
        "home_score"
    ]
].copy()

away_rows = away_rows.rename(
    columns={
        "away_team": "team",
        "home_team": "opponent",
        "away_score": "points_scored",
        "home_score": "points_allowed"
    }
)

away_rows["is_home"] = 0

In [30]:
# Combine home and away rows
team_games = pd.concat([home_rows, away_rows], ignore_index=True)

team_games = team_games.sort_values(["team", "season", "week", "gameday"]).reset_index(drop=True)

team_games.head(10)

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home
0,2023,1,2023_01_ARI_WAS,2023-09-10,ARI,WAS,16.0,20.0,0
1,2023,2,2023_02_NYG_ARI,2023-09-17,ARI,NYG,28.0,31.0,1
2,2023,3,2023_03_DAL_ARI,2023-09-24,ARI,DAL,28.0,16.0,1
3,2023,4,2023_04_ARI_SF,2023-10-01,ARI,SF,16.0,35.0,0
4,2023,5,2023_05_CIN_ARI,2023-10-08,ARI,CIN,20.0,34.0,1
5,2023,6,2023_06_ARI_LA,2023-10-15,ARI,LA,9.0,26.0,0
6,2023,7,2023_07_ARI_SEA,2023-10-22,ARI,SEA,10.0,20.0,0
7,2023,8,2023_08_BAL_ARI,2023-10-29,ARI,BAL,24.0,31.0,1
8,2023,9,2023_09_ARI_CLE,2023-11-05,ARI,CLE,0.0,27.0,0
9,2023,10,2023_10_ATL_ARI,2023-11-12,ARI,ATL,25.0,23.0,1


In [31]:
# Add result columns
team_games["point_diff"] = team_games["points_scored"] - team_games["points_allowed"]

team_games["team_won"] = (team_games["points_scored"] > team_games["points_allowed"]).astype(int)

team_games.head()

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home,point_diff,team_won
0,2023,1,2023_01_ARI_WAS,2023-09-10,ARI,WAS,16.0,20.0,0,-4.0,0
1,2023,2,2023_02_NYG_ARI,2023-09-17,ARI,NYG,28.0,31.0,1,-3.0,0
2,2023,3,2023_03_DAL_ARI,2023-09-24,ARI,DAL,28.0,16.0,1,12.0,1
3,2023,4,2023_04_ARI_SF,2023-10-01,ARI,SF,16.0,35.0,0,-19.0,0
4,2023,5,2023_05_CIN_ARI,2023-10-08,ARI,CIN,20.0,34.0,1,-14.0,0


In [32]:
# Check one team
team_games[team_games["team"] == "KC"].head(10)

,season,week,game_id,gameday,team,opponent,points_scored,points_allowed,is_home,point_diff,team_won
799,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,1,-1.0,0
800,2023,2,2023_02_KC_JAX,2023-09-17,KC,JAX,17.0,9.0,0,8.0,1
801,2023,3,2023_03_CHI_KC,2023-09-24,KC,CHI,41.0,10.0,1,31.0,1
802,2023,4,2023_04_KC_NYJ,2023-10-01,KC,NYJ,23.0,20.0,0,3.0,1
803,2023,5,2023_05_KC_MIN,2023-10-08,KC,MIN,27.0,20.0,0,7.0,1
804,2023,6,2023_06_DEN_KC,2023-10-12,KC,DEN,19.0,8.0,1,11.0,1
805,2023,7,2023_07_LAC_KC,2023-10-22,KC,LAC,31.0,17.0,1,14.0,1
806,2023,8,2023_08_KC_DEN,2023-10-29,KC,DEN,9.0,24.0,0,-15.0,0
807,2023,9,2023_09_MIA_KC,2023-11-05,KC,MIA,21.0,14.0,1,7.0,1
808,2023,11,2023_11_PHI_KC,2023-11-20,KC,PHI,17.0,21.0,1,-4.0,0


In [33]:
# Create games played before current game
team_games["games_played_before"] = team_games.groupby(["team", "season"]).cumcount()

team_games[team_games["team"] == "KC"][
    ["season", "week", "team", "opponent", "points_scored", "points_allowed", "games_played_before"]
].head(10)

,season,week,team,opponent,points_scored,points_allowed,games_played_before
799,2023,1,KC,DET,20.0,21.0,0
800,2023,2,KC,JAX,17.0,9.0,1
801,2023,3,KC,CHI,41.0,10.0,2
802,2023,4,KC,NYJ,23.0,20.0,3
803,2023,5,KC,MIN,27.0,20.0,4
804,2023,6,KC,DEN,19.0,8.0,5
805,2023,7,KC,LAC,31.0,17.0,6
806,2023,8,KC,DEN,9.0,24.0,7
807,2023,9,KC,MIA,21.0,14.0,8
808,2023,11,KC,PHI,17.0,21.0,9


In [34]:
# Create running totals before each game
team_games["points_scored_before"] = (
    team_games.groupby(["team", "season"])["points_scored"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["points_allowed_before"] = (
    team_games.groupby(["team", "season"])["points_allowed"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["point_diff_before"] = (
    team_games.groupby(["team", "season"])["point_diff"]
    .transform(lambda x: x.cumsum().shift(1))
)

team_games["wins_before"] = (
    team_games.groupby(["team", "season"])["team_won"]
    .transform(lambda x: x.cumsum().shift(1))
)

In [35]:
# Replace missing values for first games
cols_to_fill = [
    "points_scored_before",
    "points_allowed_before",
    "point_diff_before",
    "wins_before"
]

team_games[cols_to_fill] = team_games[cols_to_fill].fillna(0)

In [36]:
# Create pre-game averages
team_games["avg_points_scored_before"] = 0.0
team_games["avg_points_allowed_before"] = 0.0
team_games["avg_point_diff_before"] = 0.0
team_games["win_pct_before"] = 0.0

mask = team_games["games_played_before"] > 0

team_games.loc[mask, "avg_points_scored_before"] = (
    team_games.loc[mask, "points_scored_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_points_allowed_before"] = (
    team_games.loc[mask, "points_allowed_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "avg_point_diff_before"] = (
    team_games.loc[mask, "point_diff_before"] / team_games.loc[mask, "games_played_before"]
)

team_games.loc[mask, "win_pct_before"] = (
    team_games.loc[mask, "wins_before"] / team_games.loc[mask, "games_played_before"]
)

In [37]:
# Check the calculated stats
team_games[team_games["team"] == "KC"][
    [
        "season",
        "week",
        "team",
        "opponent",
        "points_scored",
        "points_allowed",
        "games_played_before",
        "avg_points_scored_before",
        "avg_points_allowed_before",
        "avg_point_diff_before",
        "win_pct_before"
    ]
].head(12)

,season,week,team,opponent,points_scored,points_allowed,games_played_before,avg_points_scored_before,avg_points_allowed_before,avg_point_diff_before,win_pct_before
799,2023,1,KC,DET,20.0,21.0,0,0.000000,0.000000,0.000000,0.000000
800,2023,2,KC,JAX,17.0,9.0,1,20.000000,21.000000,-1.000000,0.000000
801,2023,3,KC,CHI,41.0,10.0,2,18.500000,15.000000,3.500000,0.500000
802,2023,4,KC,NYJ,23.0,20.0,3,26.000000,13.333333,12.666667,0.666667
803,2023,5,KC,MIN,27.0,20.0,4,25.250000,15.000000,10.250000,0.750000
804,2023,6,KC,DEN,19.0,8.0,5,25.600000,16.000000,9.600000,0.800000
805,2023,7,KC,LAC,31.0,17.0,6,24.500000,14.666667,9.833333,0.833333
806,2023,8,KC,DEN,9.0,24.0,7,25.428571,15.000000,10.428571,0.857143
807,2023,9,KC,MIA,21.0,14.0,8,23.375000,16.125000,7.250000,0.750000
808,2023,11,KC,PHI,17.0,21.0,9,23.111111,15.888889,7.222222,0.777778


In [38]:
# Create home team features
home_features = team_games[team_games["is_home"] == 1][
    [
        "game_id",
        "team",
        "games_played_before",
        "avg_points_scored_before",
        "avg_points_allowed_before",
        "avg_point_diff_before",
        "win_pct_before"
    ]
].copy()

home_features = home_features.rename(
    columns={
        "team": "home_team_check",
        "games_played_before": "home_games_played_before",
        "avg_points_scored_before": "home_avg_points_scored_before",
        "avg_points_allowed_before": "home_avg_points_allowed_before",
        "avg_point_diff_before": "home_avg_point_diff_before",
        "win_pct_before": "home_win_pct_before"
    }
)

home_features.head()

,game_id,home_team_check,home_games_played_before,home_avg_points_scored_before,home_avg_points_allowed_before,home_avg_point_diff_before,home_win_pct_before
1,2023_02_NYG_ARI,ARI,1,16.000000,20.000000,-4.000000,0.000000
2,2023_03_DAL_ARI,ARI,2,22.000000,25.500000,-3.500000,0.000000
4,2023_05_CIN_ARI,ARI,4,22.000000,25.500000,-3.500000,0.250000
7,2023_08_BAL_ARI,ARI,7,18.142857,26.000000,-7.857143,0.142857
9,2023_10_ATL_ARI,ARI,9,16.777778,26.666667,-9.888889,0.111111


In [39]:
# Create away team features
away_features = team_games[team_games["is_home"] == 0][
    [
        "game_id",
        "team",
        "games_played_before",
        "avg_points_scored_before",
        "avg_points_allowed_before",
        "avg_point_diff_before",
        "win_pct_before"
    ]
].copy()

away_features = away_features.rename(
    columns={
        "team": "away_team_check",
        "games_played_before": "away_games_played_before",
        "avg_points_scored_before": "away_avg_points_scored_before",
        "avg_points_allowed_before": "away_avg_points_allowed_before",
        "avg_point_diff_before": "away_avg_point_diff_before",
        "win_pct_before": "away_win_pct_before"
    }
)

away_features.head()

,game_id,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before
0,2023_01_ARI_WAS,ARI,0,0.000,0.000000,0.000000,0.000000
3,2023_04_ARI_SF,ARI,3,24.000,22.333333,1.666667,0.333333
5,2023_06_ARI_LA,ARI,5,21.600,27.200000,-5.600000,0.200000
6,2023_07_ARI_SEA,ARI,6,19.500,27.000000,-7.500000,0.166667
8,2023_09_ARI_CLE,ARI,8,18.875,26.625000,-7.750000,0.125000


In [40]:
# Merge features onto game rows
modeling_data = games.merge(home_features, on="game_id", how="left")

modeling_data = modeling_data.merge(away_features, on="game_id", how="left")

modeling_data.head()

,season,week,game_id,gameday,home_team,away_team,home_score,away_score,home_team_won,home_point_diff,...,home_avg_points_scored_before,home_avg_points_allowed_before,home_avg_point_diff_before,home_win_pct_before,away_team_check,away_games_played_before,away_avg_points_scored_before,away_avg_points_allowed_before,away_avg_point_diff_before,away_win_pct_before
0,2023,1,2023_01_DET_KC,2023-09-07,KC,DET,20.0,21.0,0,-1.0,...,0.0,0.0,0.0,0.0,DET,0,0.0,0.0,0.0,0.0
1,2023,1,2023_01_CAR_ATL,2023-09-10,ATL,CAR,24.0,10.0,1,14.0,...,0.0,0.0,0.0,0.0,CAR,0,0.0,0.0,0.0,0.0
2,2023,1,2023_01_HOU_BAL,2023-09-10,BAL,HOU,25.0,9.0,1,16.0,...,0.0,0.0,0.0,0.0,HOU,0,0.0,0.0,0.0,0.0
3,2023,1,2023_01_CIN_CLE,2023-09-10,CLE,CIN,24.0,3.0,1,21.0,...,0.0,0.0,0.0,0.0,CIN,0,0.0,0.0,0.0,0.0
4,2023,1,2023_01_JAX_IND,2023-09-10,IND,JAX,21.0,31.0,0,-10.0,...,0.0,0.0,0.0,0.0,JAX,0,0.0,0.0,0.0,0.0


In [41]:
# Check that teams matched correctly
modeling_data[
    [
        "game_id",
        "home_team",
        "home_team_check",
        "away_team",
        "away_team_check"
    ]
].head(10)

,game_id,home_team,home_team_check,away_team,away_team_check
0,2023_01_DET_KC,KC,KC,DET,DET
1,2023_01_CAR_ATL,ATL,ATL,CAR,CAR
2,2023_01_HOU_BAL,BAL,BAL,HOU,HOU
3,2023_01_CIN_CLE,CLE,CLE,CIN,CIN
4,2023_01_JAX_IND,IND,IND,JAX,JAX
5,2023_01_TB_MIN,MIN,MIN,TB,TB
6,2023_01_TEN_NO,NO,NO,TEN,TEN
7,2023_01_SF_PIT,PIT,PIT,SF,SF
8,2023_01_ARI_WAS,WAS,WAS,ARI,ARI
9,2023_01_GB_CHI,CHI,CHI,GB,GB


In [42]:
# Drop check columns 
modeling_data = modeling_data.drop(columns=["home_team_check", "away_team_check"])

In [43]:
# Create difference features
modeling_data["avg_points_scored_diff"] = (
    modeling_data["home_avg_points_scored_before"] - modeling_data["away_avg_points_scored_before"]
)

modeling_data["avg_points_allowed_diff"] = (
    modeling_data["home_avg_points_allowed_before"] - modeling_data["away_avg_points_allowed_before"]
)

modeling_data["avg_point_diff_diff"] = (
    modeling_data["home_avg_point_diff_before"] - modeling_data["away_avg_point_diff_before"]
)

modeling_data["win_pct_diff"] = (
    modeling_data["home_win_pct_before"] - modeling_data["away_win_pct_before"]
)

In [44]:
# Preview modeling columns
modeling_data[
    [
        "season",
        "week",
        "home_team",
        "away_team",
        "home_score",
        "away_score",
        "home_team_won",
        "home_avg_points_scored_before",
        "away_avg_points_scored_before",
        "avg_points_scored_diff",
        "home_avg_points_allowed_before",
        "away_avg_points_allowed_before",
        "avg_points_allowed_diff",
        "home_win_pct_before",
        "away_win_pct_before",
        "win_pct_diff"
    ]
].head(15)

,season,week,home_team,away_team,home_score,away_score,home_team_won,home_avg_points_scored_before,away_avg_points_scored_before,avg_points_scored_diff,home_avg_points_allowed_before,away_avg_points_allowed_before,avg_points_allowed_diff,home_win_pct_before,away_win_pct_before,win_pct_diff
0,2023,1,KC,DET,20.0,21.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2023,1,ATL,CAR,24.0,10.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2023,1,BAL,HOU,25.0,9.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2023,1,CLE,CIN,24.0,3.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2023,1,IND,JAX,21.0,31.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,2023,1,MIN,TB,17.0,20.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,2023,1,NO,TEN,16.0,15.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,2023,1,PIT,SF,7.0,30.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,2023,1,WAS,ARI,20.0,16.0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,2023,1,CHI,GB,20.0,38.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [45]:
# Check for missing values
modeling_data.isna().sum().sort_values(ascending=False).head(20)

season                            0
week                              0
avg_point_diff_diff               0
avg_points_allowed_diff           0
avg_points_scored_diff            0
away_win_pct_before               0
away_avg_point_diff_before        0
away_avg_points_allowed_before    0
away_avg_points_scored_before     0
away_games_played_before          0
home_win_pct_before               0
home_avg_point_diff_before        0
home_avg_points_allowed_before    0
home_avg_points_scored_before     0
home_games_played_before          0
home_point_diff                   0
home_team_won                     0
away_score                        0
home_score                        0
away_team                         0
dtype: int64

In [46]:
# Save the modeling dataset
modeling_data.to_csv("../data/processed/modeling_dataset_2023_2025.csv", index=False)

print("Saved modeling dataset.")
print(modeling_data.shape)

Saved modeling dataset.
(855, 24)


In [47]:
# Final summary
print("Rows:", modeling_data.shape[0])
print("Columns:", modeling_data.shape[1])
print("Seasons:", modeling_data["season"].unique())
print("Home win rate:", f"{modeling_data['home_team_won'].mean():.2%}")

Rows: 855
Columns: 24
Seasons: [2023 2024 2025]
Home win rate: 54.85%


## Summary

In this notebook, I created the first set of pre-game features for the NFL forecasting model.

The main steps were:

1. Converted each game into two team-game rows.
2. Calculated team stats before each game.
3. Created pre-game averages for points scored, points allowed, point differential, and win percentage.
4. Merged those features back into one row per game.
5. Created difference features comparing the home team to the away team.
6. Saved the first modeling dataset.

The next step is to use this dataset to train a baseline prediction model.